# MNIST Classification

## Import packages

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import os
from pathlib import Path

## Set up hyperparameter

In [ ]:
batch_size = 64
test_batch_size = 1000
epochs = 10
lr = 0.01
try_cuda = True
seed = 42
logging_interval = 100 # how many batches to wait before logging
logging_dir = None

## Set up our Neural Network

In [ ]:
# Defining Architecture,loss and optimizer
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5)
        self.drop = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(1024, 1024)
        self.fc2 = nn.Linear(1024, 10)

        # Dictionary to store intermediate net inputs and activations
        self.intermediates = {}

    def forward(self, x):
        # Layer 1: conv1
        conv1_net = self.conv1(x)               # net input
        conv1_relu = F.relu(conv1_net)          # activation after ReLU
        conv1_pool = F.max_pool2d(conv1_relu, 2)# activation after Max-Pooling

        # Layer 2: conv2
        conv2_net = self.conv2(conv1_pool)      # net input
        conv2_relu = F.relu(conv2_net)          # activation after ReLU
        conv2_pool = F.max_pool2d(conv2_relu, 2)# activation after Max-Pooling

        # Flatten from 64*4*4 to 1024
        flat = conv2_pool.view(-1, 1024)

        # Layer 3: fc1
        fc1_net = self.fc1(flat)                # net input
        fc1_relu = F.relu(fc1_net)              # activation after ReLU
        fc1_drop = self.drop(fc1_relu)

        # Layer 4: fc2
        fc2_net = self.fc2(fc1_drop)            # net input
        out = F.softmax(fc2_net, dim=1)         # final output

        # Cache intermediate terms for TensorBoard monitoring
        self.intermediates = {
            'conv1': {
                'net_input': conv1_net,
                'relu': conv1_relu,
                'maxpool': conv1_pool
            },
            'conv2': {
                'net_input': conv2_net,
                'relu': conv2_relu,
                'maxpool': conv2_pool
            },
            'fc1': {
                'net_input': fc1_net,
                'relu': fc1_relu
            },
            'fc2': {
                'net_input': fc2_net
            }
        }

        return out


## Instantiate our NN

In [ ]:
model = Net()
optimizer = optim.Adam(model.parameters(), lr=lr)
writer = SummaryWriter(log_dir=logging_dir)
# Defining the test and trainig loops
eps = 1e-13

def log_tensor_statistics(writer, main_tag, tensor, step):
    # Using add_scalars combines min, max, mean, std into a single consolidated chart per tensor.
    # This prevents TensorBoard from being cluttered with dozens of separate scalar plots.
    t = tensor.detach().float()
    writer.add_scalars(f'{main_tag}_stats', {
        'min': t.min().item(),
        'max': t.max().item(),
        'mean': t.mean().item(),
        'std': t.std().item()
    }, step)
    writer.add_histogram(f'{main_tag}_hist', t.cpu().numpy(), step)


## Set up Enviorment

In [ ]:
#deciding whether to send to the cpu or not if available
if torch.cuda.is_available() and try_cuda:
    cuda = True
    torch.cuda.manual_seed(seed)
    model = model.cuda()
else:
    cuda = False
    torch.manual_seed(seed)

# Setting up data
transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.01307,), (0.3081,))
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=True, download=True,
                   transform=transform),
    batch_size=batch_size, shuffle=True)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=False, transform=transform),
    batch_size=batch_size, shuffle=True)

## Define a Training loop

In [ ]:
def train(epoch):
    model.train()

    criterion = nn.NLLLoss(reduction='none')
    total_train_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        if cuda:
            data, target = data.cuda(), target.cuda()

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(torch.log(output + eps), target) # = sum_k(-t_k * log(y_k))
        batch_loss = loss.mean() # average over the batch
        batch_loss.backward()
        optimizer.step()

        # Accumulate loss over all samples in the epoch
        total_train_loss += loss.sum().item()

        # Step index for global training steps
        step = (epoch - 1) * len(train_loader) + batch_idx

        # Monitor training loss during training (Part a)
        writer.add_scalar('loss/train_batch', batch_loss.item(), step)

        if batch_idx % logging_interval == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {batch_loss.item():.6f}')

    # Correct calculation: Epoch average loss over entire dataset
    epoch_train_loss = total_train_loss / len(train_loader.dataset)
    writer.add_scalar('loss/train_epoch', epoch_train_loss, epoch)

    # Monitor statistics (min, max, mean, standard deviation, histogram) after each epoch (Part b)
    # 1. Weights and biases (Parameters)
    for name, param in model.named_parameters():
        clean_name = name.replace('.', '_')
        log_tensor_statistics(writer, f'parameters/{clean_name}', param, epoch)

    # 2. Net inputs, activations after ReLU, and activations after Max-Pooling at each layer
    for layer_name, layer_dict in model.intermediates.items():
        for term_name, tensor_val in layer_dict.items():
            log_tensor_statistics(writer, f'activations/{layer_name}_{term_name}', tensor_val, epoch)


## Define a Testing loop


In [ ]:
def test(epoch):
    model.eval()
    test_loss = 0
    correct = 0

    criterion = nn.NLLLoss(reduction='none')

    with torch.no_grad():
        for data, target in test_loader:
            if cuda:
                data, target = data.cuda(), target.cuda()

            output = model(data)

            test_loss += criterion(torch.log(output + eps), target).sum().item() # sum up batch loss
            pred = output.argmax(dim=1) # get the index of the max log-probability
            correct += pred.eq(target).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = 100. * correct / len(test_loader.dataset)
    test_error = 100.0 - test_accuracy
    print(f'Test Epoch: {epoch}\tLoss: {test_loss:.6f}\tAccuracy: {test_accuracy:.2f}%\tTest Error: {test_error:.2f}%')

    # Monitor test metrics after each epoch
    writer.add_scalar('loss/test_epoch', test_loss, epoch)
    writer.add_scalars('test_metrics', {
        'accuracy': test_accuracy,
        'error': test_error
    }, epoch)


## Run those loops

In [ ]:

training_start_time = datetime.now()
for epoch in range(1, epochs + 1):
    train(epoch)
    test(epoch)

writer.close()

## Launch TensorBoard
Run the cell below to inspect training loss, test error, and statistics (min, max, mean, std, histogram) for weights, biases, and layer activations.

In [ ]:
# Load the TensorBoard notebook extension
%load_ext tensorboard\n
# Launch TensorBoard inside the notebook (points to log directory or default 'runs' directory)\n
%tensorboard --logdir runs\n